In [ ]:
# =====================================================================================
# Project: PTB-XL Metadata Audit  -  v0.2 death experiment
# Code version: v0.2
# Date: 2026-09-10
# Purpose: metadata-only diagnostic classification. This is a new experiment, not an
#          extension of the read-only audit (v0.1.x). It trains real models on
#          metadata alone and reports whether they beat chance at predicting the
#          diagnostic superclass, on the official patient-disjoint split.
#
# SPLIT PROTOCOL: train on strat_fold in {1..8}, evaluate on strat_fold == 9.
# strat_fold == 10 is never loaded into any array used by this cell. Two runtime
# asserts enforce this and the patient-disjointness of train vs eval.
#
# Independent variable: the feature block (B0..B5, see Section 2). Everything else
# (models, seeds, preprocessing) is fixed and pre-registered below before any score
# is seen.
#
# CPU only. No deep model. No plot. Nothing is written inside the dataset folder.
#
# Sections:
#   0   audit completion carried over from the cancelled v0.1.3:
#       0a) superclass prevalence per fold (1-9) + overall row -> validates the
#           global permutation null used in v0.1.2 Section 7.7
#       0b) fold-9 marginal prevalence per superclass -> chance AUPRC baselines
#   1   preprocessing (all decisions pre-registered here, fit on folds 1-8 only)
#   2   feature blocks (B0_intercept .. B5_all)
#   3   models: LogisticRegression (standardized features) and
#       HistGradientBoostingClassifier (raw features), each with SEED in
#       {42, 123, 2024}, no hyperparameter tuning
#   4   metrics: per-class and macro AUROC/AUPRC next to their chance baselines,
#       plus a patient-level bootstrap 95% CI for macro-AUROC (N_BOOT=2000,
#       BOOT_SEED=7)
#   5   decision table (rows = block x learner, macro-AUROC mean across seeds,
#       its bootstrap CI, delta vs B1_physio) and the B3 >= 0.60 verdict
#   6   logistic-regression coefficient inspection (B5_all, seed=42): top 15
#       |coefficient| features per superclass, on the standardized feature scale
#   7   manifest (full on disk; console prints only top-level keys with sizes)
#
# Outputs (all in OUT_DIR):
#    1) v0.2_superclass_prevalence_by_fold.csv
#    2) v0.2_fold9_chance_baseline.csv
#    3) v0.2_metrics_full.csv
#    4) v0.2_decision_table.csv
#    5) v0.2_coefficients_top15.csv
#    6) v0.2_manifest.json
# =====================================================================================

import os, sys, json, ast, hashlib, platform, time
from datetime import datetime
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- configuration ---------------------------------------
DATA_ROOT = r"D:\ptbxl-metadata-audit\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
OUT_DIR   = r"D:\ptbxl-metadata-audit\outputs\v0.2"
CODE_VERSION = "v0.2"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

SEEDS = [42, 123, 2024]
N_BOOT = 2000
BOOT_SEED = 7
FLAT_PREVALENCE_THRESHOLD_PP = 5.0   # judgment call, stated explicitly in the printout
VERDICT_THRESHOLD = 0.60
LEARNERS = ["LogisticRegression", "HistGradientBoosting"]

os.makedirs(OUT_DIR, exist_ok=True)

_root_abs, _out_abs = os.path.abspath(DATA_ROOT), os.path.abspath(OUT_DIR)
assert not _out_abs.lower().startswith(_root_abs.lower()), "OUT_DIR is inside the dataset folder - aborting."

RUN_STARTED = datetime.now().astimezone().isoformat()
OUTPUT_FILES = []
WARNINGS = []


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def save_csv(obj, name, index=True):
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=index, encoding="utf-8-sig")
    OUTPUT_FILES.append(name)
    print("[saved] " + p)
    return p


def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def note(msg):
    print("[NOTE] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def fast_auc(y, s):
    """Binary AUROC via the Mann-Whitney U statistic (average-rank tie handling).
    Used instead of sklearn.metrics.roc_auc_score for speed inside the bootstrap
    loop; both compute the identical quantity."""
    y = np.asarray(y)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = rankdata(np.asarray(s))
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def to_level_string(series, missing="__MISSING__"):
    """Human-readable, zero-padded categorical level for numeric columns (e.g.
    '007' for site) so sorting and printed feature names stay legible."""
    if pd.api.types.is_numeric_dtype(series):
        nn = series.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            return series.apply(lambda v: missing if pd.isna(v) else str(int(v)).zfill(width))
    return series.apply(lambda v: missing if pd.isna(v) else str(v).strip())


def encode_categorical(level_series, train_mask, col_name):
    """One-hot encode a pre-stringified categorical column. The category list is
    fixed from the training rows only, always includes an explicit '__MISSING__'
    level, and always includes a catch-all '__UNKNOWN__' level for anything seen
    only outside training (e.g. only in fold 9). Both splits therefore produce the
    exact same set of columns."""
    train_vals = level_series[train_mask]
    train_categories = sorted(set(train_vals.unique().tolist()) | {"__MISSING__"})
    known = set(train_categories)
    remapped = level_series.where(level_series.isin(known), other="__UNKNOWN__")
    categories = train_categories + ["__UNKNOWN__"]
    cat = pd.Categorical(remapped, categories=categories)
    dummies = pd.get_dummies(cat)
    dummies.columns = ["{}={}".format(col_name, c) for c in dummies.columns]
    dummies.index = level_series.index
    return dummies, categories


# =====================================================================================
# LOAD + PROVENANCE
# =====================================================================================
hdr("LOAD + PROVENANCE")

print("run started  :", RUN_STARTED)
print("code version :", CODE_VERSION)
print("python       :", sys.version.replace("\n", " "))
print("platform     :", platform.platform())
print("numpy        :", np.__version__, "| pandas:", pd.__version__)

DB_CSV = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV = os.path.join(DATA_ROOT, "scp_statements.csv")
for p in (DB_CSV, SCP_CSV):
    assert os.path.isfile(p), "file not found: " + p

INPUT_HASHES = {os.path.basename(p): sha256_of(p) for p in (DB_CSV, SCP_CSV)}
for k, v in INPUT_HASHES.items():
    print("{:26s} {}".format(k, v))

df_all = pd.read_csv(DB_CSV, index_col="ecg_id")
print("full dataset shape:", df_all.shape)

# --- Working set: folds 1-9 ONLY. Fold 10 is never assembled into any array below. ---
df9 = df_all[df_all["strat_fold"] <= 9].copy()
mask_train = df9["strat_fold"] <= 8
mask_eval = df9["strat_fold"] == 9
print("folds 1-8 (train) records:", int(mask_train.sum()), "| fold 9 (eval) records:", int(mask_eval.sum()))

# --- Required runtime assertions ---
assert (df9["strat_fold"] == 10).sum() == 0, \
    "fold 10 leaked into the working set - aborting."
n_fold10_in_source = int((df_all["strat_fold"] == 10).sum())
print("fold 10 records in the source file (untouched):", n_fold10_in_source,
      "| fold 10 records in the working set: 0 (by construction, verified by assert)")

train_patients = set(df9.loc[mask_train, "patient_id"])
eval_patients = set(df9.loc[mask_eval, "patient_id"])
assert train_patients.isdisjoint(eval_patients), \
    "patient-level leakage between train (folds 1-8) and eval (fold 9) - aborting."
print("train patients:", len(train_patients), "| eval patients:", len(eval_patients),
      "| disjoint: PASS")


# =====================================================================================
# LABEL DERIVATION (same primary convention as the v0.1.x audit: all scp_codes
# keys mapped via diagnostic==1, no likelihood threshold)
# =====================================================================================
scp = pd.read_csv(SCP_CSV, index_col=0)
diag = scp[scp["diagnostic"].fillna(0) == 1]
code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}
parsed = df9["scp_codes"].apply(ast.literal_eval)
rec_classes = parsed.apply(lambda d: {code2class[k] for k in d if k in code2class})
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df9.index)
for c in SUPERCLASSES:
    n_pos_train = int(lab.loc[mask_train, c].sum())
    n_pos_eval = int(lab.loc[mask_eval, c].sum())
    if n_pos_train == 0 or n_pos_eval == 0:
        warn("{}: zero positive examples in {} set - this class cannot be modeled.".format(
            c, "train" if n_pos_train == 0 else "eval"))


# =====================================================================================
# SECTION 0 - AUDIT COMPLETION (carried over from the cancelled v0.1.3)
# =====================================================================================
hdr("SECTION 0a - SUPERCLASS PREVALENCE BY FOLD (1-9)")

prev_by_fold = (lab.groupby(df9["strat_fold"]).mean() * 100)
prev_by_fold.insert(0, "n_records", lab.groupby(df9["strat_fold"]).size())
overall_row = {"n_records": len(lab)}
overall_row.update({c: float(lab[c].mean() * 100) for c in SUPERCLASSES})
prev_table = pd.concat([prev_by_fold, pd.DataFrame([overall_row], index=["__OVERALL_1to9__"])])
print(prev_table.round(3).to_string())
save_csv(prev_table, PREFIX + "superclass_prevalence_by_fold.csv", index=True)

fold_ranges = prev_by_fold[SUPERCLASSES].max() - prev_by_fold[SUPERCLASSES].min()
max_range = float(fold_ranges.max())
max_range_class = fold_ranges.idxmax()
is_flat = max_range <= FLAT_PREVALENCE_THRESHOLD_PP
print("\nmax range across folds 1-9, by superclass (pp):")
print(fold_ranges.round(3).to_frame("range_pp").to_string())
print("\nVERDICT: prevalence across folds 1-9 is {} (largest range = {:.2f}pp on {}, threshold = {:.1f}pp) "
      "-> {} the global permutation null used in v0.1.2 Section 7.7.".format(
          "FLAT" if is_flat else "NOT FLAT", max_range, max_range_class, FLAT_PREVALENCE_THRESHOLD_PP,
          "validates" if is_flat else "calls into question"))
if not is_flat:
    warn("Superclass prevalence is not flat across folds 1-9 (max range {:.2f}pp on {}) - the global "
         "permutation null in v0.1.2 Section 7.7 assumed a fold-independent label distribution.".format(
             max_range, max_range_class))

hdr("SECTION 0b - FOLD-9 CHANCE AUPRC BASELINE")
prev_fold9 = lab.loc[mask_eval].mean()
fold9_chance_df = pd.DataFrame({
    "superclass": SUPERCLASSES,
    "n_positive": [int(lab.loc[mask_eval, c].sum()) for c in SUPERCLASSES],
    "chance_auprc_prevalence": [float(prev_fold9[c]) for c in SUPERCLASSES],
}).set_index("superclass")
print("These are the chance AUPRC baselines used throughout Sections 4-5 (chance AUROC = 0.50 always):")
print(fold9_chance_df.round(6).to_string())
save_csv(fold9_chance_df, PREFIX + "fold9_chance_baseline.csv", index=True)


# =====================================================================================
# SECTION 1 - PREPROCESSING (pre-registered)
# =====================================================================================
hdr("SECTION 1 - PREPROCESSING")

age_raw = df9["age"]
age_sentinel_mask = (age_raw == 300)
n_age_sentinel = int(age_sentinel_mask.sum())
age_clean = age_raw.where(~age_sentinel_mask, np.nan)
age_censored = age_sentinel_mask.astype(float)
age_imputed = age_clean.fillna(90.0)
print("age: sentinel value 300 (ages above 89, per the v1.0.2 changelog) affects",
      n_age_sentinel, "records ({:.2f}%); replaced with NaN, flagged via age_censored, "
      "imputed with 90.".format(n_age_sentinel / len(df9) * 100))

sex = df9["sex"].astype(float)
print("sex: used as-is (0/1), no transformation.")

rd = pd.to_datetime(df9["recording_date"], errors="coerce")
recording_year = rd.dt.year.astype(float)
n_year_missing = int(recording_year.isna().sum())
if n_year_missing > 0:
    train_median_year = float(recording_year[mask_train].median())
    recording_year = recording_year.fillna(train_median_year)
    warn("{} records had an unparsable recording_date; recording_year imputed with the "
         "training-set median ({}). This decision was not in the original spec and is "
         "logged here for transparency.".format(n_year_missing, train_median_year))
else:
    print("recording_year: derived from recording_date, no missing values.")

print("height / weight: excluded entirely (68%/57% missing; physiologically impossible "
      "values such as a minimum height of 6cm and a minimum weight of 5kg).")

device_level = to_level_string(df9["device"])
site_level = to_level_string(df9["site"])
nurse_level = to_level_string(df9["nurse"])

device_dummies, device_categories = encode_categorical(device_level, mask_train, "device")
site_dummies, site_categories = encode_categorical(site_level, mask_train, "site")
nurse_dummies, nurse_categories = encode_categorical(nurse_level, mask_train, "nurse")

for name, dummies, cats in [("device", device_dummies, device_categories),
                            ("site", site_dummies, site_categories),
                            ("nurse", nurse_dummies, nurse_categories)]:
    n_unknown_active = int(dummies["{}=__UNKNOWN__".format(name)].sum())
    print("{}: {} levels from folds 1-8 (incl. __MISSING__) + 1 __UNKNOWN__ catch-all "
          "-> {} one-hot columns | __UNKNOWN__ active on {} eval records".format(
              name, len(cats) - 1, dummies.shape[1], n_unknown_active))

FEATURES = pd.concat([
    age_imputed.rename("age_imputed"),
    age_censored.rename("age_censored"),
    sex.rename("sex"),
    device_dummies, site_dummies, nurse_dummies,
    recording_year.rename("recording_year"),
], axis=1)

B1_cols = ["age_imputed", "age_censored", "sex"]
B2_cols = list(device_dummies.columns) + list(site_dummies.columns) + list(nurse_dummies.columns)
B3_cols = B1_cols + B2_cols
B4_cols = ["recording_year"]
B5_cols = B1_cols + B2_cols + B4_cols

BLOCKS = {
    "B0_intercept": [],
    "B1_physio": B1_cols,
    "B2_acquisition": B2_cols,
    "B3_physio_plus_acquisition": B3_cols,
    "B4_temporal": B4_cols,
    "B5_all": B5_cols,
}

print("\n--- feature block design matrix shapes ---")
for name, cols in BLOCKS.items():
    if name == "B0_intercept":
        print("  {:28s} train=({}, 1) [constant zero column]  eval=({}, 1)".format(
            name, int(mask_train.sum()), int(mask_eval.sum())))
    else:
        print("  {:28s} train=({}, {})  eval=({}, {})".format(
            name, int(mask_train.sum()), len(cols), int(mask_eval.sum()), len(cols)))


# =====================================================================================
# SECTION 2 - FEATURE BLOCKS (definitions printed above; used from here on)
# =====================================================================================
hdr("SECTION 2 - FEATURE BLOCK DEFINITIONS")
for name, cols in BLOCKS.items():
    print("  {:28s}: {}".format(name, "(no features)" if not cols else
                                 (", ".join(cols) if len(cols) <= 6 else
                                  ", ".join(cols[:3]) + " ... " + ", ".join(cols[-2:]) +
                                  "  ({} total)".format(len(cols)))))


# =====================================================================================
# SECTION 3 + 4 - MODELS, SEEDS, METRICS, BOOTSTRAP CI
# =====================================================================================
hdr("SECTION 3+4 - MODELS x SEEDS x METRICS")

y_train_full = lab.loc[mask_train]
y_eval_full = lab.loc[mask_eval]

# --- precompute patient-level bootstrap resamples once; reused for every config ---
eval_patient_arr = df9.loc[mask_eval, "patient_id"].to_numpy()
unique_eval_patients = np.unique(eval_patient_arr)
n_eval_patients = len(unique_eval_patients)
patient_row_list = [np.where(eval_patient_arr == p)[0] for p in unique_eval_patients]

t0 = time.time()
rng_boot = np.random.default_rng(BOOT_SEED)
BOOT_ROWS = []
for _ in range(N_BOOT):
    samp = rng_boot.integers(0, n_eval_patients, size=n_eval_patients)
    BOOT_ROWS.append(np.concatenate([patient_row_list[i] for i in samp]))
print("precomputed {} patient-level bootstrap resamples of the fold-9 eval set "
      "({} patients, {} records) in {:.2f}s".format(N_BOOT, n_eval_patients, int(mask_eval.sum()),
                                                     time.time() - t0))

METRIC_ROWS = []
FITTED_LR_B5_ALL = {}
FITTED_LR_B5_ALL_FEATURE_NAMES = None

t_all = time.time()
for block_name, cols in BLOCKS.items():
    if block_name == "B0_intercept":
        Xtr_raw = np.zeros((int(mask_train.sum()), 1))
        Xev_raw = np.zeros((int(mask_eval.sum()), 1))
    else:
        Xtr_raw = FEATURES.loc[mask_train, cols].to_numpy(dtype=float)
        Xev_raw = FEATURES.loc[mask_eval, cols].to_numpy(dtype=float)

    scaler = StandardScaler().fit(Xtr_raw)
    Xtr_lr = scaler.transform(Xtr_raw)
    Xev_lr = scaler.transform(Xev_raw)

    for learner in LEARNERS:
        for seed in SEEDS:
            scores_eval = {}
            models_this = {}
            per_class_auroc, per_class_auprc = [], []

            for sc in SUPERCLASSES:
                ytr = y_train_full[sc].to_numpy()
                yev = y_eval_full[sc].to_numpy()

                if learner == "LogisticRegression":
                    model = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs", random_state=seed)
                    model.fit(Xtr_lr, ytr)
                    pos_col = list(model.classes_).index(1)
                    proba = model.predict_proba(Xev_lr)[:, pos_col]
                else:
                    model = HistGradientBoostingClassifier(random_state=seed)
                    model.fit(Xtr_raw, ytr)
                    pos_col = list(model.classes_).index(1)
                    proba = model.predict_proba(Xev_raw)[:, pos_col]

                scores_eval[sc] = proba
                models_this[sc] = model

                auroc = fast_auc(yev, proba)
                auprc = float(average_precision_score(yev, proba)) if (0 < yev.sum() < len(yev)) else float("nan")
                per_class_auroc.append(auroc)
                per_class_auprc.append(auprc)

                METRIC_ROWS.append({
                    "block": block_name, "learner": learner, "seed": seed, "class": sc,
                    "auroc": auroc, "auroc_chance": 0.5,
                    "auprc": auprc, "auprc_chance": float(prev_fold9[sc]),
                    "ci_low": float("nan"), "ci_high": float("nan"),
                })

            if block_name == "B5_all" and learner == "LogisticRegression" and seed == 42:
                FITTED_LR_B5_ALL = models_this
                FITTED_LR_B5_ALL_FEATURE_NAMES = cols

            macro_auroc = float(np.nanmean(per_class_auroc))
            macro_auprc = float(np.nanmean(per_class_auprc))
            macro_auprc_chance = float(np.mean([prev_fold9[c] for c in SUPERCLASSES]))

            boot_macros = np.empty(N_BOOT)
            for bi, rows in enumerate(BOOT_ROWS):
                accs = []
                for sc in SUPERCLASSES:
                    yb = y_eval_full[sc].to_numpy()[rows]
                    sb = scores_eval[sc][rows]
                    a = fast_auc(yb, sb)
                    if not np.isnan(a):
                        accs.append(a)
                boot_macros[bi] = np.mean(accs) if accs else np.nan
            n_boot_valid = int(np.sum(~np.isnan(boot_macros)))
            ci_low, ci_high = np.nanpercentile(boot_macros, [2.5, 97.5])

            METRIC_ROWS.append({
                "block": block_name, "learner": learner, "seed": seed, "class": "MACRO",
                "auroc": macro_auroc, "auroc_chance": 0.5,
                "auprc": macro_auprc, "auprc_chance": macro_auprc_chance,
                "ci_low": float(ci_low), "ci_high": float(ci_high),
            })
            print("  {:28s} | {:22s} | seed={:5d} | macro-AUROC={:.4f} (chance 0.50) "
                  "95%CI[{:.4f},{:.4f}] n_boot_valid={} | macro-AUPRC={:.4f} (chance {:.4f})".format(
                      block_name, learner, seed, macro_auroc, ci_low, ci_high, n_boot_valid,
                      macro_auprc, macro_auprc_chance))

print("\ntotal model-fitting + bootstrap time: {:.1f}s".format(time.time() - t_all))

metrics_df = pd.DataFrame(METRIC_ROWS)
save_csv(metrics_df, PREFIX + "metrics_full.csv", index=False)

print("\n--- per-class metrics, folds combined into one tidy table (head) ---")
print(metrics_df.head(20).round(5).to_string(index=False))


# =====================================================================================
# SECTION 5 - DECISION TABLE
# =====================================================================================
hdr("SECTION 5 - DECISION TABLE")

macro_df = metrics_df[metrics_df["class"] == "MACRO"]
decision_rows = []
for block_name in BLOCKS:
    for learner in LEARNERS:
        sub = macro_df[(macro_df["block"] == block_name) & (macro_df["learner"] == learner)]
        decision_rows.append({
            "block": block_name, "learner": learner,
            "macro_auroc_mean": float(sub["auroc"].mean()),
            "macro_auroc_std_across_seeds": float(sub["auroc"].std(ddof=0)),
            "ci_low_mean": float(sub["ci_low"].mean()),
            "ci_high_mean": float(sub["ci_high"].mean()),
        })
decision_df = pd.DataFrame(decision_rows).set_index(["block", "learner"])

b1_ref = {learner: decision_df.loc[("B1_physio", learner), "macro_auroc_mean"] for learner in LEARNERS}
decision_df["delta_vs_B1_physio"] = [
    row["macro_auroc_mean"] - b1_ref[idx[1]] for idx, row in decision_df.iterrows()
]

print(decision_df.round(5).to_string())
print("\nNote: macro_auroc_mean / ci_*_mean average the 3 per-seed point estimates and CI "
      "bounds. LogisticRegression uses lbfgs (deterministic given fixed data), so its "
      "std across seeds should be ~0; any residual is float-precision noise, not real "
      "seed sensitivity.")
save_csv(decision_df.reset_index(), PREFIX + "decision_table.csv", index=False)

print("\n--- verdict: does B3_physio_plus_acquisition reach macro-AUROC >= {:.2f} on fold 9? ---".format(
    VERDICT_THRESHOLD))
VERDICT = {}
for learner in LEARNERS:
    val = decision_df.loc[("B3_physio_plus_acquisition", learner), "macro_auroc_mean"]
    passed = bool(val >= VERDICT_THRESHOLD)
    VERDICT[learner] = {"macro_auroc_mean": float(val), "passes_threshold": passed}
    print("  [{}] macro-AUROC = {:.4f} -> {}".format(learner, val, "PASS" if passed else "FAIL"))
overall_pass = any(v["passes_threshold"] for v in VERDICT.values())
print("overall: {} (at least one learner reaches the threshold)".format(
    "PASS" if overall_pass else "FAIL"))
VERDICT["overall_pass"] = overall_pass


# =====================================================================================
# SECTION 6 - COEFFICIENT INSPECTION (LogisticRegression, B5_all, seed=42)
# =====================================================================================
hdr("SECTION 6 - COEFFICIENT INSPECTION (B5_all, LogisticRegression, seed=42)")
print("Coefficients are on the STANDARDIZED feature scale (StandardScaler fit on "
      "folds 1-8), i.e. per-1-SD change in each feature - not directly comparable to "
      "raw units.\n")

coef_rows = []
for sc in SUPERCLASSES:
    model = FITTED_LR_B5_ALL[sc]
    coefs = model.coef_.ravel()
    order = np.argsort(-np.abs(coefs))[:15]
    print("--- " + sc + " ---")
    sub_rows = []
    for rank, idx in enumerate(order, start=1):
        sub_rows.append({"rank": rank, "feature": FITTED_LR_B5_ALL_FEATURE_NAMES[idx],
                         "coefficient": float(coefs[idx]), "abs_coefficient": float(abs(coefs[idx]))})
        coef_rows.append({"superclass": sc, **sub_rows[-1]})
    print(pd.DataFrame(sub_rows).set_index("rank").round(4).to_string())
    print()

coef_df = pd.DataFrame(coef_rows)
save_csv(coef_df, PREFIX + "coefficients_top15.csv", index=False)


# =====================================================================================
# SECTION 7 - MANIFEST
# =====================================================================================
hdr("SECTION 7 - MANIFEST")

PREPROCESSING_DECISIONS = {
    "age_sentinel_value": 300,
    "age_sentinel_meaning": "ages above 89, per the v1.0.2 changelog",
    "age_imputed_with": 90.0,
    "n_age_sentinel_affected": n_age_sentinel,
    "sex_encoding": "used as-is (0/1)",
    "categorical_columns": ["device", "site", "nurse"],
    "categorical_missing_level": "__MISSING__",
    "categorical_unseen_level": "__UNKNOWN__",
    "categorical_encoder_fit_on": "folds 1-8 only",
    "recording_year_source": "recording_date, integer year",
    "n_recording_year_missing": n_year_missing,
    "excluded_columns": ["height", "weight"],
    "excluded_reason": "68%/57% missing; physiologically impossible values "
                       "(min height 6cm, min weight 5kg)",
    "lr_feature_scaling": "StandardScaler fit on the training block only, applied to "
                          "all columns including one-hot indicators",
    "hgb_feature_scaling": "none (tree-based)",
    "label_convention": "all scp_codes keys mapped via diagnostic==1, no likelihood threshold "
                        "(same primary convention as the v0.1.x audit)",
}

manifest = {
    "code_version": CODE_VERSION,
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "dataset_root": DATA_ROOT,
    "output_dir": OUT_DIR,
    "input_files_sha256": INPUT_HASHES,
    "split_protocol": {
        "train_folds": "1-8", "eval_fold": 9, "fold10_touched": False,
        "n_fold10_records_in_source_untouched": n_fold10_in_source,
        "n_train_records": int(mask_train.sum()), "n_eval_records": int(mask_eval.sum()),
        "n_train_patients": len(train_patients), "n_eval_patients": len(eval_patients),
    },
    "seeds": SEEDS, "n_boot": N_BOOT, "boot_seed": BOOT_SEED,
    "flat_prevalence_threshold_pp": FLAT_PREVALENCE_THRESHOLD_PP,
    "verdict_threshold": VERDICT_THRESHOLD,
    "preprocessing_decisions": PREPROCESSING_DECISIONS,
    "feature_blocks": BLOCKS,
    "superclass_prevalence_by_fold": prev_table.reset_index().rename(
        columns={"index": "fold"}).to_dict(orient="records"),
    "prevalence_flat_across_folds": bool(is_flat),
    "prevalence_max_range_pp": max_range,
    "fold9_chance_baseline": fold9_chance_df.reset_index().to_dict(orient="records"),
    "metrics": METRIC_ROWS,
    "decision_table": decision_df.reset_index().to_dict(orient="records"),
    "verdict": VERDICT,
    "coefficients_model": "B5_all, LogisticRegression, seed=42, standardized feature scale",
    "checks": {
        "fold10_excluded_by_construction": True,
        "train_eval_patient_disjoint": True,
    },
    "warnings": WARNINGS,
    "outputs": [],
}

for name in OUTPUT_FILES:
    p = os.path.join(OUT_DIR, name)
    manifest["outputs"].append({"file": name, "bytes": os.path.getsize(p), "sha256": sha256_of(p)})

MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=_json_default)

print("[saved] " + MANIFEST_PATH)
print("\n--- manifest top-level keys (console shows sizes only; the file on disk is complete) ---")
for k, v in manifest.items():
    size_chars = len(json.dumps(v, ensure_ascii=False, default=_json_default))
    print("  {:36s} size={} chars".format(k, size_chars))
print("\nmanifest path  :", MANIFEST_PATH)
print("manifest sha256:", sha256_of(MANIFEST_PATH))


# =====================================================================================
# FINAL SUMMARY
# =====================================================================================
hdr("SUMMARY - " + CODE_VERSION)
print("train / eval records      :", int(mask_train.sum()), "/", int(mask_eval.sum()))
print("train / eval patients      :", len(train_patients), "/", len(eval_patients))
print("fold 10 touched            : No (0 records; assert verified)")
print("prevalence flat 1-9        :", "YES" if is_flat else "NO", "(max range {:.2f}pp)".format(max_range))
print("\ndecision table (macro-AUROC mean across seeds):")
print(decision_df[["macro_auroc_mean", "ci_low_mean", "ci_high_mean", "delta_vs_B1_physio"]]
      .round(4).to_string())
print("\nB3 >= {:.2f} verdict        :".format(VERDICT_THRESHOLD),
      {k: v["passes_threshold"] for k, v in VERDICT.items() if k != "overall_pass"},
      "-> overall:", "PASS" if VERDICT["overall_pass"] else "FAIL")
print("files produced             :", len(OUTPUT_FILES) + 1)
print("\nwarnings ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)

print("\n" + CODE_VERSION + " finished successfully.")